In [20]:
import pandas as pd 
import numpy as np
import torch 
import torch.nn as nn
from torch.utils.data import Dataset,DataLoader 
from collections import Counter
import string 
import re 
import emoji


In [21]:
# Stach all columns to one column
df = pd.read_csv('/kaggle/input/datasets/jerryqu/reddit-conversations/casual_data_windows.csv')
df = df.drop(columns=["Unnamed: 0"])
df = df.stack()
df = df.reset_index(name="values").drop(columns=['level_1','level_0'])
df.head()


,values
0,What kind of phone(s) do you guys have?
1,I have a pixel. It's pretty great. Much better...
2,Does it really charge all the way in 15 min?
3,I have a pixel. It's pretty great. Much better...
4,Does it really charge all the way in 15 min?


In [22]:
df.duplicated().sum()

np.int64(96310)

In [23]:
# Remove duplicate quetions 
df.drop_duplicates(inplace=True)
df.duplicated().sum()

np.int64(0)

In [24]:
# Pre-Process text function 
exclude = string.punctuation
def pre_processtext(text):
    
    # Lowercasing 
    text = str(text).lower()
    
    # Remove HTML tags
    pattern = re.compile('<.*?>')
    text = pattern.sub(r'',text)

    # Remove emojis
    text = emoji.replace_emoji(text,replace='')

    # Remove Punctuations
    text = text.translate(str.maketrans('','',exclude))

    # Return as list
    return [word for word in text.split(" ") if word != ""]


df['values']=df['values'].apply(pre_processtext)

In [25]:
df["values"]

0             [what, kind, of, phones, do, you, guys, have]
1         [i, have, a, pixel, its, pretty, great, much, ...
2         [does, it, really, charge, all, the, way, in, ...
5         [pretty, fast, ive, never, timed, it, but, its...
8         [cool, ive, been, thinking, of, getting, one, ...
                                ...                        
168873    [are, you, from, singaporebecause, i, recognis...
168874    [its, also, in, the, philippines, but, our, pr...
168877                                [also, in, australia]
168880                                  [and, new, zealand]
168886                      [nope, i’m, from, new, zealand]
Name: values, Length: 72577, dtype: object

In [26]:
# Build vocabulary
from itertools import chain

vocab = {'<UNK>':0,'<PAD>':1}

word_counts = Counter(chain.from_iterable(df['values']))

for word,count in word_counts.items():
    if word not in vocab and count >= 3: #removed uncommon words
        vocab[word] = len(vocab)

In [27]:
len(vocab)

9762

In [28]:
# Function for donverting each word with its value index from vocabulary
def conver_indieces(text,vocab):
    
    # Set index for each wrod in text
    text = [vocab.get(w,vocab['<UNK>']) for w in text]
    
    # Saves the possible combination for each sentence 
    input_data = [text[:i+1] for i in range(1,len(text))]
    
    return input_data


df['values']=df['values'].apply(conver_indieces,args=(vocab,))

In [29]:
# Faltten dataset for Making X and Y supervised data  
df_flat = pd.DataFrame(df['values'].explode())
df_flat = df_flat.dropna()
df_flat.head()

,values
0,"[2, 3]"
0,"[2, 3, 4]"
0,"[2, 3, 4, 5]"
0,"[2, 3, 4, 5, 6]"
0,"[2, 3, 4, 5, 6, 7]"


In [30]:
X_list = [x[:-1] for x in df_flat['values']]
y_list = [x[-1] for x in df_flat['values']]

In [31]:
# Dataset Class 
class MyCustomDataset(Dataset):
    
    def __init__(self,X_list,y_list):
        self.features = X_list
        self.labels = y_list
    
    def __len__(self):
        return len(self.features)
    
    def __getitem__(self,index):
        x = torch.tensor(self.features[index], dtype=torch.long)
        y = torch.tensor(self.labels[index], dtype=torch.long)
        return x, y

In [32]:
train_dataset = MyCustomDataset(X_list,y_list)

In [33]:
from torch.nn.utils.rnn import pad_sequence

# Add Padding as batch wise with padding value as in vocab
def collate_fn(batch):
    feature,labels = zip(*batch)
    feature_padded = pad_sequence(feature,batch_first=True,padding_value=1)
    labels_stacked = torch.stack(labels)
    return feature_padded,labels_stacked
    
# Making DataLoader for traindata
train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True,collate_fn=collate_fn,pin_memory=True)

In [34]:
class MyModel(nn.Module):
    
    def __init__(self,vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size,embedding_dim=128)
        self.lstm = nn.LSTM(128,256,batch_first=True,num_layers=2,dropout=0.3)
        self.fc = nn.Linear(256,vocab_size)
        
    def forward(self,x):
        embedded = self.embedding(x)
        intermidiate_hidden_state,(final_hidden_state,final_cell_state) = self.lstm(embedded)
        result = self.fc(final_hidden_state[-1])
        return result

In [35]:
# Defining model and port on GPU
model = MyModel(len(vocab))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)
model.to(device)

cuda


MyModel(
  (embedding): Embedding(9762, 128)
  (lstm): LSTM(128, 256, num_layers=2, batch_first=True, dropout=0.3)
  (fc): Linear(in_features=256, out_features=9762, bias=True)
)

In [36]:
# defining parameters 
learning_rate=0.001
epochs = 50

# loss function here ignore padding for each output 
creterion = nn.CrossEntropyLoss(ignore_index=1)

# Optimizer 
optimizer = torch.optim.Adam(model.parameters(),lr=learning_rate)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=2,
)

In [37]:
# Traning Loop 
for i in range(epochs):
    total_loss=0
    for batch_features,batch_labels in train_loader:
        batch_features,batch_labels = batch_features.to(device),batch_labels.to(device)
        
        #Making gradiants Zero
        optimizer.zero_grad()

        # Forward Pass
        output=model(batch_features)

        # Calculate Loss 
        loss=creterion(output,batch_labels)

        # Backword pass 
        loss.backward()

        # Gradiant clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(),max_norm=1.0)

        # Updating parameters of model
        optimizer.step()

        # Calculate total loss  
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    scheduler.step(avg_loss)

    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch:{i+1}, Loss:{avg_loss:.4f}, LR:{current_lr}")

Epoch:1, Loss:6.0269, LR:0.001
Epoch:2, Loss:5.4256, LR:0.001
Epoch:3, Loss:5.2246, LR:0.001
Epoch:4, Loss:5.0987, LR:0.001
Epoch:5, Loss:5.0002, LR:0.001
Epoch:6, Loss:4.9181, LR:0.001
Epoch:7, Loss:4.8499, LR:0.001
Epoch:8, Loss:4.7823, LR:0.001
Epoch:9, Loss:4.7227, LR:0.001
Epoch:10, Loss:4.6713, LR:0.001
Epoch:11, Loss:4.6233, LR:0.001
Epoch:12, Loss:4.5804, LR:0.001
Epoch:13, Loss:4.5398, LR:0.001
Epoch:14, Loss:4.4998, LR:0.001
Epoch:15, Loss:4.4641, LR:0.001
Epoch:16, Loss:4.4302, LR:0.001
Epoch:17, Loss:4.3999, LR:0.001
Epoch:18, Loss:4.3697, LR:0.001
Epoch:19, Loss:4.3424, LR:0.001
Epoch:20, Loss:4.3162, LR:0.001
Epoch:21, Loss:4.2924, LR:0.001
Epoch:22, Loss:4.2689, LR:0.001
Epoch:23, Loss:4.2499, LR:0.001
Epoch:24, Loss:4.2278, LR:0.001
Epoch:25, Loss:4.2089, LR:0.001
Epoch:26, Loss:4.1915, LR:0.001
Epoch:27, Loss:4.1728, LR:0.001
Epoch:28, Loss:4.1565, LR:0.001
Epoch:29, Loss:4.1388, LR:0.001
Epoch:30, Loss:4.1197, LR:0.001
Epoch:31, Loss:4.1034, LR:0.001
Epoch:32, Loss:4.

In [38]:
def predict_word(text,model,vocab,top_k=5):
    
    # Created reverse of current vocab 
    indx_to_word = {idx:word for word,idx in vocab.items()}

    # Process Tokens exactly like training data
    tokens = pre_processtext(text)
    
    # Convert to indices 
    indices = [vocab.get(token,vocab['<UNK>']) for token in tokens]
    
    # Convert to tensor with adding dimention so it looks (1,number of tokens)
    input_tensor = torch.tensor(indices,dtype=torch.long).unsqueeze(0).to(device)

    # Forward Pass
    model.eval()
    with torch.no_grad():
        output = model(input_tensor)
        probs = torch.softmax(output,dim=-1)
        top_probs,top_indices = torch.topk(probs,k=top_k,dim=-1)

    print(f"INPUT: '{text}'")
    print(f"Top {top_k} Prediction for next word:\n")
    for i in range(top_k):
        word = indx_to_word[top_indices[0][i].item()]
        prob = top_probs[0][i].item()
        print(f" {i+1}. '{word}' - {prob*100:.2f}%")

In [51]:
predict_word("can u tell me",model,vocab,top_k=3)

INPUT: 'can u tell me'
Top 3 Prediction for next word:

 1. 'about' - 33.90%
 2. 'that' - 10.62%
 3. 'the' - 6.64%
